## 라이브러리, 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from statsmodels.stats.outliers_influence import variance_inflation_factor

# 한글 폰트 설정 (OS별로 사용 가능한 폰트가 다르므로 순서대로 시도, 없으면 기본 폰트 사용)
import matplotlib.font_manager as fm

def set_korean_font():
    candidates = ['Malgun Gothic', 'AppleGothic', 'NanumGothic']
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in available:
            plt.rcParams['font.family'] = name
            return name
    return None  # 한글 폰트가 없으면 라벨이 깨질 수 있음 (그래프 자체는 정상 출력)

set_korean_font()
plt.rcParams['axes.unicode_minus'] = False

# 데이터 로드 (이 저장소 기준 상대 경로)
DATA_PATH = 'data/fnc_final_target.xlsx'
df = pd.read_excel(DATA_PATH)

# 253 x 27
# 253개 앨범(2019.10 ~ 2026.06 발매), 39개 그룹(아이돌, 밴드, 솔로 포함)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 결측치

구조적 결측(e.g. 데뷔 앨범의 직전 초동 판매량 = 0) 파악

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'결측 개수': missing, '결측률(%)': missing_pct})
missing_df[missing_df['결측 개수'] > 0]

- 직전 초동 판매량이 42%로 가장 높은 결측 → 데뷔 앨범, 초동 판매 정보 없음 등의 사유로 인한 구조적 결측
- SNS 관련 지표(Spotify, TikTok, Instagram 등)도 15~20% 결측 → 그룹별 활동 시점 차이에서 기인
- 이전 컴백기간 콘서트 관련 변수 결측 → 콘서트 미개최 그룹 존재

## 타겟 분포

In [ ]:
target = df['실제 초동 판매량']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 원본 분포
axes[0].hist(target, bins=40, color='#4A6FA5', edgecolor='white')
axes[0].set_title('실제 초동 판매량 분포 (원본)', fontweight='bold')
axes[0].set_xlabel('초동 판매량')
axes[0].set_ylabel('빈도')
axes[0].axvline(target.median(), color='red', linestyle='--', label=f'중앙값: {int(target.median()):,}')
axes[0].axvline(target.mean(), color='orange', linestyle='--', label=f'평균: {int(target.mean()):,}')
axes[0].legend()

# 로그 변환 분포
axes[1].hist(np.log1p(target), bins=40, color='#6B8E23', edgecolor='white')
axes[1].set_title('실제 초동 판매량 분포 (log1p 변환)', fontweight='bold')
axes[1].set_xlabel('log(초동 판매량 + 1)')
axes[1].set_ylabel('빈도')

plt.tight_layout()
plt.show()

print(f"최솟값: {target.min():,}")
print(f"최댓값: {target.max():,}")
print(f"중앙값: {int(target.median()):,}")
print(f"평균:   {int(target.mean()):,}")
print(f"왜도(skewness): {target.skew():.2f}")

- right-skewed된 초동 판매량 → 대다수 앨범이 20만 장 이하 or 소수 대형 앨범이 100만 장 이상

## 범주형 변수와 초동 판매량

In [ ]:
cat_vars = ['앨범 타입_국내외', '앨범 타입_음반 종류', '성별', '그룹 타입']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

palette = ['#4A6FA5', '#6B8E23', '#B85450', '#DAA520', '#8B7B8B']

for i, col in enumerate(cat_vars):
    order = df.groupby(col)['실제 초동 판매량'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='실제 초동 판매량', ax=axes[i],
                order=order, palette=palette[:len(order)])
    axes[i].set_title(f'{col}별 초동 판매량', fontweight='bold')
    axes[i].set_yscale('log')
    axes[i].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# 그룹별 요약 통계
for col in cat_vars:
    print(f'\n =================')
    summary = df.groupby(col)['실제 초동 판매량'].agg(['count', 'median', 'mean']).round(0).astype(int)
    summary.columns = ['앨범 수', '중앙값', '평균']
    print(summary.sort_values('중앙값', ascending=False).to_string())

- 국내 발매 앨범이 해외 발매 대비 판매량 중앙값이 뚜렷하게 높음
- 앨범 종류 중 미니앨범 판매량이 가장 높고, 디지털 싱글은 상대적으로 낮음
- 그룹 타입에 따라서는, 아이돌 그룹이 밴드/솔로 대비 초동 판매량 높음

## 수치형 변수와 타겟의 상관관계

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_with_target = df[num_cols].corr(method='spearman')['실제 초동 판매량'].drop('실제 초동 판매량').sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#B85450' if x < 0 else '#4A6FA5' for x in corr_with_target.values]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='white')
ax.set_title('수치형 변수와 초동 판매량의 상관관계', fontsize=13, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.5)
for i, v in enumerate(corr_with_target.values):
    ax.text(v + (0.01 if v >= 0 else -0.01), i, f'{v:.2f}',
            va='center', ha='left' if v >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

- 팬덤 자산 지표(직전 초동 판매량, Instagram/Spotify/TikTok 팔로워)가 상위
- 콘텐츠 반응 지표(유튜브 티저 조회수, 직전 뮤비 조회수) 양의 상관
- 연차, 발매월 등 시점 변수는 약한 상관 → 비선형 효과 가능성

## 변수 간 상관관계

In [ ]:
# SNS 지표군 다중공선성
sns_cols = ['Spotify 월간 청취자', 'Spotify 팔로워', 'TikTok 팔로워', '구독자', 'Instagram 팔로워', '실제 초동 판매량']
sns_corr = df[sns_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(sns_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('SNS 지표 상관관계', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

- SNS 지표 5개 ("Spotify 월간 청취자", "Spotify 팔로워", "TikTok 팔로워", "구독자", "Instagram 팔로워") 간 매우 강한 상관 → 다중공선성
-> SNS 지표 중 결측이 가장 적고 타겟 상관이 높은 Instagram 팔로워만 유지

In [ ]:
# VIF 진단
sns_vif_cols = ['Spotify 월간 청취자', 'Spotify 팔로워', 'TikTok 팔로워', '구독자', 'Instagram 팔로워']
vif_df = df[sns_vif_cols].dropna()

vif_result = pd.DataFrame({
    '변수': sns_vif_cols,
    'VIF': [variance_inflation_factor(vif_df.values, i) for i in range(len(sns_vif_cols))]
}).sort_values('VIF', ascending=False)

print(vif_result.to_string(index=False))

- 상당수 SNS 지표의 VIF가 임계값(5~10)을 초과하며 다중공선성 확인되므로 Instagram 팔로워 남기고 삭제

## 발매 시점 분석

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 발매월별 판매량
month_stats = df.groupby('발매월')['실제 초동 판매량'].median().reset_index()
axes[0].bar(month_stats['발매월'], month_stats['실제 초동 판매량'], color='#4A6FA5', edgecolor='white')
axes[0].set_title('발매월별 초동 판매량 (중앙값)', fontweight='bold')
axes[0].set_xlabel('발매월')
axes[0].set_ylabel('초동 판매량 중앙값')
axes[0].set_xticks(range(1, 13))

# 연차별 판매량
year_stats = df.groupby('연차')['실제 초동 판매량'].median().reset_index()
axes[1].plot(year_stats['연차'], year_stats['실제 초동 판매량'],
             marker='o', color='#6B8E23', linewidth=2)
axes[1].set_title('데뷔 연차별 초동 판매량 (중앙값)', fontweight='bold')
axes[1].set_xlabel('연차')
axes[1].set_ylabel('초동 판매량 중앙값')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

- 상반기(1~5월) 발매가 하반기 대비 판매량 중앙값이 높음 → 시즌 효과 존재
- 데뷔 초반 연차에 성장 → 중반 정점 → 후반 완만한 하락하는 패턴 다수

## 그룹별 초동 판매량

In [ ]:
group_stats = df.groupby('그룹명')['실제 초동 판매량'].agg(['count', 'median']).sort_values('median', ascending=False)
group_stats.columns = ['앨범 수', '판매량 중앙값']

top15, bottom15 = group_stats.head(15), group_stats.tail(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(top15.index[::-1], top15['판매량 중앙값'][::-1], color='#4A6FA5', edgecolor='white')
axes[0].set_title('Top 15 그룹 (초동 판매량 중앙값)', fontweight='bold')

axes[1].barh(bottom15.index[::-1], bottom15['판매량 중앙값'][::-1], color='#B85450', edgecolor='white')
axes[1].set_title('Bottom 15 그룹 (초동 판매량 중앙값)', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n분석 그룹 수: {df["그룹명"].nunique()}개')
print(f'그룹당 평균 앨범 수: {group_stats["앨범 수"].mean():.1f}개')

## 앨범 구성(트랙 수) vs 초동 판매량

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df[df['트랙 수'] <= 12], x='트랙 수', y='실제 초동 판매량',
            ax=ax, color='#4A6FA5')
ax.set_yscale('log')
ax.set_title('트랙 수별 초동 판매량 분포', fontweight='bold')
plt.tight_layout()
plt.show()

# 트랙 수 구간별 요약
df['트랙 수 구간'] = pd.cut(df['트랙 수'], bins=[0, 2, 5, 8, 100],
                             labels=['1-2곡', '3-5곡', '6-8곡', '9곡+'])
print(df.groupby('트랙 수 구간')['실제 초동 판매량'].agg(['count', 'median']).astype(int).to_string())

- 3~5곡 구성(미니앨범 규모)에서 판매량 중앙값이 가장 높음
- 1~2곡 싱글은 상대적으로 낮고, 9곡 이상은 오히려 감소 → 미니앨범 포맷의 최적점 확인

## 콘서트 이력 & 컴백 주기 분석

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 콘서트 개최 여부
df['콘서트 개최 여부'] = np.where(df['이전 컴백기간 콘서트 횟수'].fillna(0) > 0, '개최', '미개최')
sns.boxplot(data=df, x='콘서트 개최 여부', y='실제 초동 판매량', ax=axes[0], palette=['#B85450', '#4A6FA5'])
axes[0].set_yscale('log')
axes[0].set_title('이전 컴백기간 콘서트 개최 여부별 판매량', fontweight='bold')

# 컴백 주기
df_cycle = df.dropna(subset=['컴백 주기'])
df_cycle['컴백 주기 구간'] = pd.cut(df_cycle['컴백 주기'], bins=[0, 60, 120, 200, 400, 2000], labels=['~2개월', '2-4개월', '4-6.5개월', '6.5-13개월', '13개월 초과'])
sns.boxplot(data=df_cycle, x='컴백 주기 구간', y='실제 초동 판매량', ax=axes[1], color='#6B8E23')
axes[1].set_yscale('log')
axes[1].set_title('컴백 주기별 초동 판매량', fontweight='bold')

plt.tight_layout()
plt.show()

- 이전 컴백기간에 콘서트를 개최한 그룹은 초동 판매량이 뚜렷하게 높음 → 팬덤 활성화 효과
- 컴백 주기는 너무 짧거나(2개월 이내) 너무 길면(13개월 초과) 판매량 저하 경향 -> 4~13개월 주기가 가장 안정적